# Chromate-Dichromate Equilibrium
$$2CrO_4^{2-}(aq)+2H^+(aq)\rightleftharpoons Cr_2O_7^{2-}(aq)+H_2O(aq)$$
This is a pH-dependent equilibrium between the yellow chromate ion ($CrO_4^{2-}$) and the orange dichromate ion ($Cr_2O_7^{2-}$).

This reaction happens in two stages involving an intermediate species: hydrogen chromate ($HCrO_4^-$).
1. $CrO_4^{2-}+H^+\rightleftharpoons HCrO_4^-$
2. $2HCrO_4^-\rightleftharpoons Cr_2O_7^{2-}+H_2O$

If a strong alkali ($OH^-$) is injected into the system after it has reached equilibrium, an irreversible neutralisation reaction with the hydrogen ions in the system can be modelled:
$$H^+(aq)+OH^-(aq)\rightarrow H_2O(l)$$
This has two effects on the primary equilibrium:
* It removes a reactant ($H^+$).
* It produces a product ($H_2O$).

The equilibrium will shift to the left, causing the concentration of the orange dichromate ion to decrease, while the concentration of the yellow chromate ion increases.

In [ ]:
import numpy as np
from chemical_engine import ChemicalSpecies, Reaction, ChemicalSystem
from visualisation import create_interface

CrO4 = ChemicalSpecies('CrO4(2-)', phase='aqueous') 
HCrO4 = ChemicalSpecies('HCrO4(-)', phase='aqueous') # intermediate
Cr2O7 = ChemicalSpecies('Cr2O7(2-)', phase='aqueous') 
H_ion = ChemicalSpecies('H+', phase='aqueous')
OH_ion = ChemicalSpecies('OH-', phase='aqueous')
H2O = ChemicalSpecies('H2O', phase='solvent', density=1000.0, molar_mass=18.015)

r_prot_fwd = Reaction({'CrO4(2-)': 1, 'H+': 1}, {'HCrO4(-)': 1}, A=1.0e7, Ea=0.0)
r_prot_rev = Reaction({'HCrO4(-)': 1}, {'CrO4(2-)': 1, 'H+': 1}, A=1.0e4, Ea=0.0)

r_dimer_fwd = Reaction({'HCrO4(-)': 2}, {'Cr2O7(2-)': 1, 'H2O': 1}, A=1.0e5, Ea=0.0)
r_dimer_rev = Reaction({'Cr2O7(2-)': 1, 'H2O': 1}, {'HCrO4(-)': 2}, A=5.0e2, Ea=0.0)

r_neut_fwd = Reaction({'H+': 1, 'OH-': 1}, {'H2O': 1}, A=1.0e10, Ea=0.0)
r_neut_rev = Reaction({'H2O': 1}, {'H+': 1, 'OH-': 1}, A=1.0e-6, Ea=0.0)

initial_moles = {
    'CrO4(2-)': 0.1,   
    'H+': 0.2,       # excess acid
    'HCrO4(-)': 0.0, 
    'Cr2O7(2-)': 0.0,
    'H2O': 0.0,   
    'OH-': 0.0      
}

overall_chromate = Reaction(
    reactants={'CrO4(2-)': 2, 'H+': 2},
    products={'Cr2O7(2-)': 1, 'H2O': 1},
    A=0, Ea=0
)

system = ChemicalSystem(
    species_list=[CrO4, HCrO4, Cr2O7, H_ion, OH_ion, H2O],
    reaction_list=[r_prot_fwd, r_prot_rev, r_dimer_fwd, r_dimer_rev, r_neut_fwd, r_neut_rev], 
    initial_moles=initial_moles,
    initial_V=1.0,  
    initial_T=298.0, 
    method='Radau', 
    rtol=1e-8,      
    atol=1e-11, overall_reaction=overall_chromate
)

create_interface(system)

# Haber Process

In [ ]:
import numpy as np
N2 = ChemicalSpecies('N2', vdw_a=1.352, vdw_b=0.0387)
H2 = ChemicalSpecies('H2', vdw_a=0.244, vdw_b=0.0266)
NH3 = ChemicalSpecies('NH3', vdw_a=4.170, vdw_b=0.0371)

R = 8.314
T_ref = 900

Ea_f = 85000  # J/mol
A_f = 0.1 * np.exp(Ea_f / (R * T_ref))

Ea_r = 177000 # J/mol
A_r = 2.5 * np.exp(Ea_r / (R * T_ref))

r_fwd = Reaction(
    reactants={'N2': 1, 'H2': 3}, 
    products={'NH3': 2}, 
    A=A_f, 
    Ea=Ea_f
)

r_rev = Reaction(
    reactants={'NH3': 2}, 
    products={'N2': 1, 'H2': 3}, 
    A=A_r, 
    Ea=Ea_r
)

initial_moles = {'N2': 1.0, 'H2': 3.0, 'NH3': 0.0}

overall_haber = Reaction(
    reactants={'N2': 1, 'H2': 3},
    products={'NH3': 2},
    A=0, Ea=0 
)

haber_system = ChemicalSystem(
    species_list=[N2, H2, NH3],
    reaction_list=[r_fwd, r_rev],
    initial_moles=initial_moles,
    initial_V=1.0,
    initial_T=700, # typical industrial temperature
    overall_reaction=overall_haber
)

create_interface(haber_system)

# SN1 vs E1 Competition
This models the hydrolysis of tert-butyl chloride, an example of where a single intermediate can follow two different pathways depending on the energy available in the system (temperature).

## Mechanism
The reaction proceeds in two distinct stages.

**Ionisation (rate-determining step):** the leaving group ($Cl^-$) departs, leaving behind a carbocation intermediate. This step is slow and reversible.
$$t-\text{Bu}Cl\rightleftharpoons t-\text{Bu}^++Cl^-$$

**Substitution:** water acts as a nucleophile, attaching to the central carbon.
$$t-\text{Bu}^++2H_2O\rightarrow t-\text{BuOH}+H_3O^+$$
This reaction has a lower activation energy. At low temperatures, molecules have limited kinetic energy. They take the "path of least resistance", leading to the substitution product, and so this pathway dominates at lower temperatures.

**Elimination:** water acts as a base, stealing a proton from a methyl group, causing the formation of a double bond.
$$t-\text{Bu}^++H_2O\rightarrow C_4H_8+H_3O^+$$
This reaction has a higher activation energy as a lot of energy is required to break C-H bonds, but it has a higher pre-exponential factor as it increases entropy. This path dominates at higher temperatures.

## Other Observations
On the graph, $[t-\text{Bu}^+]$ (the intermediate) will remain low and constant throughout the reaction. This is because the carbocation is highly unstable - as soon as it forms, it is consumed. ($d[I]/dt\approx0$).

In [ ]:
# substrate
s_tbucl = ChemicalSpecies("tBuCl", phase='aqueous', molar_mass=92.57)

# solvent/nucleophile (SN1)/base (E1)
s_h2o   = ChemicalSpecies("H2O", phase='solvent', density=1000, molar_mass=18.015)

# intermediates and ions
s_carbocation   = ChemicalSpecies("tBu+", phase='aqueous') # carbocation
s_cl    = ChemicalSpecies("Cl-", phase='aqueous')
s_h     = ChemicalSpecies("H3O+", phase='aqueous')

# products
s_tbuoh = ChemicalSpecies("tBuOH", phase='aqueous') # substitution product
s_isobut= ChemicalSpecies("Isobutylene", phase='gas') # elimination product

species_list = [s_tbucl, s_h2o, s_carbocation, s_cl, s_h, s_tbuoh, s_isobut]

# ionisation (RDS)
r_rds_fwd = Reaction(
    reactants={'tBuCl': 1},
    products={'tBu+': 1, 'Cl-': 1},
    A=2.0e10, Ea=70000.0 # slow step
)
# reverse of step 1
r_rds_rev = Reaction(
    reactants={'tBu+': 1, 'Cl-': 1},
    products={'tBuCl': 1},
    A=5.0e10, Ea=10000.0 # fast reverse
)

# SN1
# lower Ea
r_sn1 = Reaction(
    reactants={'tBu+': 1, 'H2O': 1},
    products={'tBuOH': 1, 'H3O+': 1},
    A=1.0e8, Ea=15000.0 
)

# E1
# higher Ea, entropically favoured (higher A)
r_e1 = Reaction(
    reactants={'tBu+': 1, 'H2O': 1},
    products={'Isobutylene': 1, 'H3O+': 1},
    A=8.0e11, Ea=45000.0 
)

reactions = [r_rds_fwd, r_rds_rev, r_sn1, r_e1]

# initial moles
initial_moles = {
    'tBuCl': 1.0,
    'H2O': 0.0, # auto-calculated by solvent density
    'tBu+': 0.0,
    'Cl-': 0.0,
    'H3O+': 0.0,
    'tBuOH': 0.0,
    'Isobutylene': 0.0
}

competing_sys = ChemicalSystem(
    species_list, reactions, initial_moles, 
    initial_V=1.0, initial_T=298.0
)

create_interface(competing_sys)

# The Brusselator

In [ ]:
A = ChemicalSpecies('A', species_type='pool', phase='aqueous')
B = ChemicalSpecies('B', species_type='pool', phase='aqueous')
X = ChemicalSpecies('X', phase='aqueous')
Y = ChemicalSpecies('Y', phase='aqueous')
P = ChemicalSpecies('P', phase='aqueous') # P is a product sink

A_param = 1.0
B_param = 3.0
scaling_factor = 0.1 # slows down the reaction to make it easier to solve

r1 = Reaction({'A': 1}, {'X': 1}, A=1.0 * scaling_factor, Ea=0)
r2 = Reaction({'X': 2, 'Y': 1}, {'X': 3}, A=1.0 * scaling_factor, Ea=0)
r3 = Reaction({'B': 1, 'X': 1}, {'Y': 1, 'P': 1}, A=1.0 * scaling_factor, Ea=0)
r4 = Reaction({'X': 1}, {'P': 1}, A=1.0 * scaling_factor, Ea=0)

initial_moles_brusselator = {
    'A': A_param,
    'B': B_param,
    'X': 1.5,
    'Y': 3.0,
    'P': 0.0
}

brusselator_system = ChemicalSystem(
    species_list=[A, B, X, Y, P],
    reaction_list=[r1, r2, r3, r4],
    initial_moles=initial_moles_brusselator,
    initial_V=1.0,
    initial_T=300,
    method='BDF',     
    rtol=1e-4,     
    atol=1e-7
)

create_interface(brusselator_system)

# Buffer Solutions

In [ ]:
R_J_MOL_K = 8.314
water = ChemicalSpecies("H2O", species_type='pool', phase='solvent', density=1000.0, molar_mass=18.015)

# ions
proton = ChemicalSpecies("H3O+", phase='aqueous', charge=1)
hydroxide = ChemicalSpecies("OH-", phase='aqueous', charge=-1)
sodium = ChemicalSpecies("Na+", phase='aqueous', charge=1) # spectator ion

weak_acid = ChemicalSpecies("HA", phase='aqueous')
conj_base = ChemicalSpecies("A-", phase='aqueous', charge=-1)

weak_base = ChemicalSpecies("NH3", phase='aqueous')
conj_acid = ChemicalSpecies("NH4+", phase='aqueous', charge=+1)

species_list = [water, proton, hydroxide, sodium, weak_acid, conj_base, weak_base, conj_acid]

T_ref = 298.0
RT = R_J_MOL_K * T_ref

# water ionisation
# 2 H2O <-> H3O+ + OH-
Ea_r_w = 12000.0
k_r_w = 1.3e11
A_r_w = k_r_w / np.exp(-Ea_r_w / RT)

Ea_f_w = 68000.0
k_f_w = 4.21e-7
A_f_w = k_f_w / np.exp(-Ea_f_w / RT)

# weak acid dissociation
Ea_r_a = 10000.0
k_r_a = 5.0e10
A_r_a = k_r_a / np.exp(-Ea_r_a / RT)

Ea_f_a = 10000.0 
k_f_a = 1.56e4
A_f_a = k_f_a / np.exp(-Ea_f_a / RT)

# weak base dissociation
Ea_r_b = 10000.0
k_r_b = 3.0e10
A_r_b = k_r_b / np.exp(-Ea_r_b / RT)

Ea_f_b = 10000.0
k_f_b = 9.6e3
A_f_b = k_f_b / np.exp(-Ea_f_b / RT)

r1_fwd = Reaction({'H2O': 2}, {'H3O+': 1, 'OH-': 1}, A=A_f_w, Ea=Ea_f_w)
r1_rev = Reaction({'H3O+': 1, 'OH-': 1}, {'H2O': 2}, A=A_r_w, Ea=Ea_r_w)

r2_fwd = Reaction({'HA': 1, 'H2O': 1}, {'A-': 1, 'H3O+': 1}, A=A_f_a, Ea=Ea_f_a)
r2_rev = Reaction({'A-': 1, 'H3O+': 1}, {'HA': 1, 'H2O': 1}, A=A_r_a, Ea=Ea_r_a)

r3_fwd = Reaction({'NH3': 1, 'H2O': 1}, {'NH4+': 1, 'OH-': 1}, A=A_f_b, Ea=Ea_f_b)
r3_rev = Reaction({'NH4+': 1, 'OH-': 1}, {'NH3': 1, 'H2O': 1}, A=A_r_b, Ea=Ea_r_b)

reactions = [r1_fwd, r1_rev, r2_fwd, r2_rev, r3_fwd, r3_rev]

initial_moles = {
    'H2O': 0.0, # will be auto-calculated
    'HA': 0.1,  # 0.1 moles in 1L = 0.1M
    'A-': 0.0,
    'H3O+': 0.0,
    'OH-': 0.0,
    'Na+': 0.0,
    'NH3': 0.0,
    'NH4+': 0.0
}

V_initial = 1.0 # L
T_initial = 298.0 

buffer_system = ChemicalSystem(
    species_list, reactions, initial_moles, V_initial, T_initial,
    method='Radau', rtol=1e-5, atol=1e-7, system_type='acid_base'
)

print(f"pKa {4.76}")
print(f"pH expected: { -np.log10(np.sqrt(1.74e-5 * 0.1)) :.2f}") 

create_interface(buffer_system)